# 05b — Pseudo-Labeling (Self-Training) — ELECTRA-small

Iteratively fine-tunes `utils.config.CLASSIFIER_MODEL_NAME_ALT` (ELECTRA-small)
on the 5%-per-class labeled seed (~15/class on master_data.csv, much
thinner than AG News's ~1,500/class), predicts on the full unlabeled pool,
and absorbs high-confidence predictions each round, targeting 98%
coverage. Starts from the same confidence threshold `05_pseudo_labeling.ipynb`
(DistilBERT) converged on, so the two notebooks stay as comparable as
possible; if round 0 stalls (0 labels absorbed), Step 2 re-runs at a lower
threshold, following the same tuning methodology.

**Result: complete stall, even more lenient than DistilBERT.** With only 16
labeled rows (1 example per class), round 0 absorbs exactly **0**
pseudo-labels at 0.50 (DistilBERT's converged threshold) and again at
0.35 — ELECTRA-small fine-tuned on 1 example/class never reaches even a
0.35 softmax confidence on any unlabeled row. **Final reported threshold:
0.35** (the lower value it was actually retuned to, since the shared
0.50 starting point also stalled). The final model (trained on the 16-row
seed only) scores 6.25% test accuracy — exactly the 16-class random-guess
baseline. See `history` and
`docs/superpowers/plans/2026-08-15-autolabel-notebooks.md` "Final Run"
note for the underlying labeled-seed-size context.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.metrics import evaluate_label_quality, evaluate_semisupervised
from utils.modeling import get_predictions, pseudo_label_loop
from utils.samples import save_full_output, save_label_samples

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

overlap = set(unlabeled_df["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled seed: {len(labeled_df)} | Unlabeled pool: {len(unlabeled_df)} | Test: {len(test_clean)}")

Labeled seed: 16 | Unlabeled pool: 303 | Test: 80


In [3]:
CONFIDENCE_THRESHOLD = 0.35  # 0.50 (DistilBERT's converged value, shared starting point) stalled at round 0; 0.35 kept as the final reported threshold (also stalls, see markdown above)

final_model, final_tokenizer, current_labeled, history = pseudo_label_loop(
    labeled_df, unlabeled_df,
    model_name=config.CLASSIFIER_MODEL_NAME_ALT,
    confidence_threshold=CONFIDENCE_THRESHOLD, epochs=3,
    target_coverage=0.98, max_iterations=10)

for h in history:
    print(h)

Total sample: 319 | target coverage: 98% | confidence threshold: 0.35


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


{'iteration': 0, 'total_sample': 319, 'new_labels': 0, 'new_labels_pct_of_total': 0.0, 'labeled_size': 16, 'coverage': 0.050156739811912224, 'unlabeled_size': 303}


In [4]:
pseudo_only = current_labeled.iloc[len(labeled_df):]
merged = pseudo_only.merge(unlabeled_df[["text", "true_label"]], on="text", how="left")
unresolved = unlabeled_df[~unlabeled_df["text"].isin(pseudo_only["text"])]

label_quality = evaluate_label_quality(
    true_labels=merged["true_label"].to_numpy(),
    pseudo_labels=merged["label"].to_numpy())
print("Pseudo-label quality (ELECTRA-small):", label_quality)

save_label_samples(
    merged["text"], merged["label"].to_numpy(), merged["true_label"].to_numpy(),
    config.CLASS_NAMES, n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_pseudo_labeling_electra_train_pool.csv")

full_pool_texts = pd.concat([merged["text"], unresolved["text"]], ignore_index=True)
full_pool_predicted = pd.concat(
    [merged["label"], pd.Series(-1, index=unresolved.index)], ignore_index=True)
full_pool_true = pd.concat([merged["true_label"], unresolved["true_label"]], ignore_index=True)
full_pool_summary = pd.concat([merged["summary"], unresolved["summary"]], ignore_index=True)

save_full_output(
    full_pool_texts, full_pool_predicted.to_numpy(), full_pool_true.to_numpy(), config.CLASS_NAMES,
    extra_columns={"summary": full_pool_summary.tolist()},
    path=config.RESULTS_DIR / "full_labels_pseudo_labeling_electra_train_pool.csv")
print("Saved sample + full-row train-pool outputs for pseudo_labeling_electra.")

Pseudo-label quality (ELECTRA-small): {'Label Accuracy': 0.0, 'Label Macro F1': 0.0, 'Coverage': 0.0}
Saved sample + full-row train-pool outputs for pseudo_labeling_electra.


In [5]:
test_probs = get_predictions(final_model, final_tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

semisup_results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_pseudo_labeling_electra.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_pseudo_labeling_electra.json", "w") as f:
    json.dump({"test_metrics": semisup_results, "label_quality": label_quality,
               "history": history, "confidence_threshold": CONFIDENCE_THRESHOLD}, f, indent=2)
print("Saved pseudo-labeling (ELECTRA-small) results.")

save_label_samples(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(),
    config.CLASS_NAMES, confidence=test_probs.max(axis=1), n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_pseudo_labeling_electra_test.csv")
save_full_output(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(), config.CLASS_NAMES,
    confidence=test_probs.max(axis=1), extra_columns={"summary": test_clean["summary"].tolist()},
    path=config.RESULTS_DIR / "full_labels_pseudo_labeling_electra_test.csv")
print("Saved sample + full-row test outputs for pseudo_labeling_electra.")

C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0

                precision    recall  f1-score   support

ARTS & CULTURE       0.00      0.00      0.00         5
      BUSINESS       0.00      0.00      0.00         5
        COMEDY       0.00      0.00      0.00         5
         CRIME       0.08      0.60      0.14         5
     EDUCATION       0.03      0.20      0.05         5
 ENTERTAINMENT       0.00      0.00      0.00         5
   ENVIRONMENT       0.00      0.00      0.00         5
        HEALTH       0.00      0.00      0.00         5
         MEDIA       0.00      0.00      0.00         5
          NEWS       0.00      0.00      0.00         5
      POLITICS       0.00      0.00      0.00         5
      RELIGION       0.00      0.00      0.00         5
       SCIENCE       0.00      0.00      0.00         5
        SPORTS       0.00      0.00      0.00         5
          TECH       0.00      0.00      0.00         5
         WOMEN       0.20      0.20      0.20         5

      accuracy                           0.06 